In [1]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LinearRegression

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir("../")

In [2]:
data_path = "data/"
output_dir = "model"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [3]:
min_Eg=1.5
max_Eg=1.7
max_TF=1.1
min_TF=0.9
key_cols = ['xA', 'yA', 'zA', 'xC', 'yC', 'zC']
calc_cols = ['rA', 'rB', 'rC', "TF"]
var_cols = [ "predicted"]
models = [
    ("Linear Regression", LinearRegression()),
   ("Random Forest Regression", RandomForestRegressor(max_depth=3, random_state=0)),### fits worse the right lower corner, and lower edge (confirm)
   ("Random Forest Regression1", RandomForestRegressor(max_depth=3, random_state=0)),
    ("Gradient Boosting Regression", GradientBoostingRegressor(random_state=0)),###in agreement with random forest
    ("Kernel Ridge Regression", KernelRidge())## ? also in agreement with random forest. check how it works
]

In [4]:
i=0
for name, model in models:
    model_dir = os.path.join(output_dir, name.replace(' ', '_'))
    pred_dir = os.path.join(model_dir, "prediction")
    if not os.path.exists(pred_dir):
        os.makedirs(pred_dir)


    df = pd.read_csv(
        os.path.join(pred_dir, "final_selection_"+str(min_Eg)+"_"+str(max_Eg)+"_"+str(min_TF)+".csv"))
    df = df[key_cols + calc_cols + var_cols]
    df.columns = key_cols + calc_cols + [ "pred_"+name.replace(' ', '_')]
    if i==0:
        df_fin=df
    else:
        df_fin=df_fin.merge(df, on=key_cols+calc_cols, how="outer")
    i=i+1


In [5]:
df_fin.pred_Linear_Regression.isnull().sum()

3402

In [6]:
len(df_fin)

4805

In [7]:
for col in key_cols:
    df_fin = df_fin[(df_fin[col]*10).astype(int)/10==df_fin[col]]

In [8]:
df_fin

,xA,yA,zA,xC,yC,zC,rA,rB,rC,TF,pred_Linear_Regression,pred_Random_Forest_Regression,pred_Random_Forest_Regression1,pred_Gradient_Boosting_Regression,pred_Kernel_Ridge_Regression
0,0.0,0.0,1.0,0.0,0.0,1.0,2.53,1.19,2.200,0.986612,NaN,1.537385,1.537556,NaN,NaN
2,0.0,0.0,1.0,0.0,0.1,0.9,2.53,1.19,2.176,0.988605,1.502808,1.538151,1.538322,1.529898,1.553267
4,0.0,0.0,1.0,0.0,0.2,0.8,2.53,1.19,2.152,0.990627,1.575684,1.538801,1.538322,1.555835,1.616960
6,0.0,0.0,1.0,0.0,0.3,0.7,2.53,1.19,2.128,0.992677,1.649658,1.665836,1.660844,1.645835,1.680660
14,0.0,0.0,1.0,0.1,0.0,0.9,2.53,1.19,2.161,0.989865,1.600098,1.537385,1.538322,1.507369,1.606955
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4788,0.5,0.0,0.5,0.1,0.2,0.7,2.10,1.19,2.113,0.901920,NaN,1.675166,NaN,1.694359,NaN
4793,0.5,0.0,0.5,0.2,0.0,0.8,2.10,1.19,2.122,0.901390,NaN,1.626603,1.658994,1.669313,NaN
4795,0.5,0.0,0.5,0.2,0.1,0.7,2.10,1.19,2.098,0.902808,NaN,1.675360,NaN,1.685879,NaN
4799,0.5,0.0,0.5,0.3,0.0,0.7,2.10,1.19,2.083,0.903705,NaN,1.677300,NaN,1.691045,NaN


In [9]:
df_fin.sort_values(["xC", "yC", "zC", "xA", "yA", "zA"]).to_csv("data/DFT_selection1_Sona.csv")